# Bibliotecas

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
from psycopg2 import sql

import tomllib

import pandas as pd

import pathlib
from pathlib import Path

import datetime
from datetime import date

# def localizar_raiz_projeto
    - Irá organizar os arquivos do projeto

In [ ]:
def localizar_raiz_projeto():

    pasta_atual = Path.cwd().resolve()

    for pasta in [
        pasta_atual,
        *pasta_atual.parents
    ]:

        if (
            (pasta / "requirements.txt").exists()
            and
            (pasta / "src").exists()
        ):
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )

PROJETO_RH = localizar_raiz_projeto()

CAMINHO_CONFIG = (
    PROJETO_RH /
    "config.toml"
)

print(
    f"Raiz do projeto: {PROJETO_RH}"
)

print(
    f"Config encontrado: {CAMINHO_CONFIG.exists()}"
)

# def conexao_db
    - Serve para abrir a conexao com o banco de dados

In [ ]:
def conexao_db():

    base_dir = Path(__file__).resolve().parent.parent
    config_path = base_dir / "config.toml"

    # Lê as config do banco
    with open(config_path, "rb") as arquivo:
        config = tomllib.load(arquivo)

        db = config["joaquina"]


        conn = psycopg2.connect(
            host = db["host"],
            port = db["port"],
            dbname = db["dbname"],
            user = db["user"],
            password = db["password"]
        )

        cursor = conn.cursor()

        return conn, cursor

# def tratamento_de_datas

In [ ]:
def tratamento_de_datas(valor):

    """
    Converte datas para um formato aceito pelo PostgreSQL.

    Aceita:
    - 25/09/2005
    - 25-09-2005
    - 2005-09-25
    - 2005/09/25
    - datetime/date/pandas Timestamp

    Retorna:
    - date válido
    - None quando estiver vazio, NaN, NaT ou inválido
    """
        
    if valor is None:
        return None
    
    if pd.isne(valor):
        return None
    
    if isinstance(valor, pd.Timestamp):
        return valor.date()
    
    if isinstance(valor, date):
        return valor()
    
    valor = str(valor).strip()

    if valor == "":
        return None
    
    formatos = [
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%Y-%m-%d",
        "%Y/%m/%d"
    ]

    for formato in formatos:
        try:
            return datetime.strptime(valor, formato).date()
        except ValueError:
            continue

    return None

# def tratar_valor_id

In [ ]:
def tratar_valor_id(valor):
    if pd.isna(valor):
        return None
    
    valor = str(valor).strip()

    if valor =="":
        return None
    
    return valor

# def atualizar_upsert

In [ ]:
def atualizar_upsert(
    tabela,
    coluna,
    valor,
    conn,
    cursor
):
    
    valor = tratar_valor_id(valor)

    if valor is None:
        return None
    
    query = sql.SQL("""
        INSERT INTO `{tabela} ({coluna})
        VALUES (%s)
        ON CONFLICT ({coluna})
        DO UPDATE SET {coluna} = EXCLUDED.{coluna}
        RETURNING id
    """).format(
            tabela = sql.Identifier(tabela),
            coluna = sql.Identifier(coluna)
    )

    cursor.execute(query, (valor))

    resultado = cursor.fetchone()

    if resultado:
        return resultado[0]
    
    return None

# Obter dados SQL

## def obeter_sql_dados_servidor

In [ ]:
def obeter_sql_dados_do_servidor():

    sql_dados_do_servidor = """
    INSERT INTO dados_do_servidor (
    matricula,
    nome,
    fk_cargo,
    fk_lotacao,
    admissao,
    aposentadoria,
    entrada_lotacao,
    termino,
    desligamento,
    fk_regime
    )
    VALUES %s
    ON CONFLICT 
        (matricula)
    DO UPDATE SET
        nome = EXCLUDED.nome,
        fk_cargo = EXCLUDED.fk_cargo,
        fk_lotacao = EXCLUDED.fk_lotacao,
        admissao = EXCLUDED.admissao,
        aposentadoria = EXCLUDED.aposentadoria,
        entrada_lotacao = EXCLUDED.entrada_lotacao,
        termino = EXCLUDED.termino,
        desligamento = EXCLUDED.desligamento,
        fk_regime = EXCLUDED.fk_regime
    RETURNING id;
    """

    return sql_dados_do_servidor

# Obter os IDs das chaves primarias

## def obter_ids_servidor

In [ ]:
def obter_ids_servidor(linha, conn, cursor):

    id_regime_servidor = atualizar_upsert(
                tabela = "pk_regime_servidor",
                coluna = "regime",
                valor = linha["regime"],
                conn = conn,
                cursor = cursor
    )

    id_lotacoes = atualizar_upsert(
                tabela = "pk_lotacoes",
                coluna = "lotacao"
    )

    id_cargos = atualizar_upsert()